## 0. Authenticating to GCP & Setup

In [14]:
from google.colab import auth

project_id = 'creditwise-493809'
bucket_name = 'codeml26'

!gcloud config set project {project_id}

Updated property [core/project].


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# published rent increase guidelines, %
# TAL (Quebec): rate for unheated dwellings, i.e. heating paid by the tenant; guideline: Ontario
QUEBEC_TAL_PCT        = {2017: 1.2, 2018: 1.4, 2019: 1.2, 2020: 1.2, 2021: 0.8, 2022: 1.28, 2023: 2.3, 2024: 4.0, 2025: 5.9, 2026: 3.1}
ONTARIO_GUIDELINE_PCT = {2017: 1.5, 2018: 1.8, 2019: 1.8, 2020: 2.2, 2021: 0.0, 2022: 1.2,  2023: 2.5, 2024: 2.5, 2025: 2.5, 2026: 2.1}

FORECAST_YEAR = 2026
MIN_GAP_YEARS, MAX_GAP_YEARS = 0.5, 2.5   # comparable lease pairs
HISTORY_YEARS = range(2020, 2026)

##1. Defining the Question

We will predict the same-unit effective rent increase for 2026, averaged over all units whose lease ends in 2026.

As such, we can reduce the effect of new buildings entering the dataset and accurately forecast the rent that Jadco will collect in 2026.

Finally, to understand this number better, we will be breaking it down into renewals vs relocations, apartment type (bedrooms and bathrooms), and Québec vs Ontario.

## 2. Analyzing the Data
We start with initialization: we import the pandas module and read in the provided Jadco data.

In [ ]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from pathlib import Path

# SETUP
# ensures common columns are read in consistently
data_standards = {
    'sUnitCode': 'string',
    'sBuilding': 'string'
}

# the bucket is being reorganized: use the first folder where the file exists
def bucket_file(name, folders=('jadco-data', 'jadco-participants', 'external-data')):
  for folder in folders:
    path = Path('./codeml26') / folder / name
    if path.exists():
      return path
  raise FileNotFoundError(f'{name} not found in ./codeml26/{folders}')

# initial read-in of data
# notably, we omit concession data
listings = pd.read_csv(bucket_file('equinoxe_listings.csv'), dtype=data_standards)
lease_history = pd.read_csv(bucket_file('equinoxe_lease_history.csv'), dtype=data_standards)
asking_history = pd.read_csv(bucket_file('equinoxe_asking_history.csv'), dtype=data_standards)

building_locations = {
    'Daniel-Johnson': ['Chomedey', 'Laval_full', 'Montreal'],
    'Le Carlyle': ['Mont_royal', 'Island_full', 'Montreal'],
    'Levesque': ['Chomedey', 'Laval_full', 'Montreal'],
    'Saint-Elzear': ['Chomedey', 'Laval_full', 'Montreal'],
    'The Met': ['Ottawa_downtown', 'Ottawa_full'],
    'Westpark': ['Baie_durfe', 'Island_full', 'Montreal']
}

Now it is time to begin preprocessing our data! The first step is to ensure all of our string times are processed as dates, for accurate and reliable comparison.

In [16]:
# the columns in the source data we want to parse as dates
lease_history_date_cols = ['sSignDate', 'sLeaseFrom', 'sLeaseTo']
listings_date_cols = ['sAvailable']
asking_history_date_cols = ['sMonth']

# lease history
for col in lease_history_date_cols:
  lease_history[col] = pd.to_datetime(lease_history[col], format='%Y-%m-%d')

# listings
for col in listings_date_cols:
  listings[col] = pd.to_datetime(listings[col], format='%Y-%m-%d')

# asking history
for col in asking_history_date_cols:
  # this one is encoded in a slightly different format!
  asking_history[col] = pd.to_datetime(asking_history[col], format='%Y-%m')


Next, we will filter the data down to only the columns we need.

In [17]:
lease_history_filtered = lease_history.filter(items=['hUnit', 'sBuilding', 'sUnitCode', 'sCity', 'sState', 'sRent', 'sBeds', 'sBaths', 'sSqft', 'sRentEffective', 'sSignDate', 'sLeaseFrom', 'sLeaseTo', 'sTermSeq', 'sFloor'])
listings_filtered = listings.filter(items=[          'hUnit', 'sBuilding', 'sUnitCode', 'sCity', 'sState', 'sRent', 'sBeds', 'sBaths', 'sSqft',                   'sAvailable',                          'sTermSeq', 'sFloor'])
asking_history_filtered = asking_history.filter(items=['hUnit', 'sMonth', 'sAskingRent'])

Because we are provided the asking price for the 2026 listings, we want to have each apartment in the lease history be associated with its preceeding "ask", up to one year in the past.

In [18]:
# for each entry in the lease history, get the 'ask' that occurred right before
recency = relativedelta(years=1)
for lease in lease_history_filtered.itertuples(index=True):
  unit = lease.hUnit
  date = lease.sSignDate
  year_prior = date - recency
  prev_asks = asking_history[(asking_history['hUnit'] == unit) & (asking_history['sMonth'] <= date) & (asking_history['sMonth'] >= year_prior)]
  ask = None
  if len(prev_asks) > 0:
    ask = prev_asks.loc[prev_asks['sMonth'].idxmax(), 'sAskingRent']
  lease_history_filtered.at[lease.Index, 'hAskingRent'] = ask

# same process for listings
for lease in listings_filtered.itertuples(index=True):
  unit = lease.hUnit
  date = lease.sAvailable
  year_prior = date - recency
  prev_asks = asking_history[(asking_history['hUnit'] == unit) & (asking_history['sMonth'] <= date) & (asking_history['sMonth'] >= year_prior)]
  ask = None
  if len(prev_asks) > 0:
    ask = prev_asks.loc[prev_asks['sMonth'].idxmax(), 'sAskingRent']
  listings_filtered.at[lease.Index, 'hAskingRent'] = ask

In [19]:
# FINAL PROCESSING
# one hot encoding
lease_history_encoded = pd.get_dummies(
    lease_history_filtered,
    columns=["sBuilding", "sState", "sCity"],
    dtype=int
)

listings_encoded = pd.get_dummies(
    listings_filtered,
    columns=["sBuilding", "sState", "sCity"],
    dtype=int
)

# drop sUnitCode
lease_history_trimmed = lease_history_encoded.drop(columns=['sUnitCode'])
listings_trimmed = listings_encoded.drop(columns=['sUnitCode'])

# finalize
lease_history_final = lease_history_trimmed
listings_final = listings_trimmed

lease_history_final.to_csv('./codeml26/processed/processed_lease_data.csv', index=False)
listings_final.to_csv('./codeml26/processed/processed_listing_data.csv', index=False)

In [20]:
# dump newly processed files
!gcloud storage cp -r  ./codeml26/processed/ gs://codeml26/

Copying file://./codeml26/processed/processed_lease_data.csv to gs://codeml26/processed/processed_lease_data.csv
Copying file://./codeml26/processed/processed_listing_data.csv to gs://codeml26/processed/processed_listing_data.csv

Average throughput: 7.6MiB/s


## 3. Same-unit growth

In [ ]:
# read the raw lease history from the bucket
!gcloud storage cp gs://codeml26/jadco-data/equinoxe_lease_history.csv . || gcloud storage cp gs://codeml26/jadco-participants/equinoxe_lease_history.csv .
leases = pd.read_csv('./equinoxe_lease_history.csv', dtype={'sUnitCode': str}, parse_dates=['sLeaseFrom', 'sLeaseTo'])
leases['year'] = leases['sLeaseFrom'].dt.year

In [ ]:
# get lease information prior to current lease (same unit = sPropCode + sUnitCode)
UNIT_KEY = ['sPropCode', 'sUnitCode']
pairs = leases.sort_values(UNIT_KEY + ['sLeaseFrom'], kind='stable')
grouped = pairs.groupby(UNIT_KEY)
pairs['prev_rent']       = grouped['sRent'].shift(1)
pairs['prev_rent_eff']   = grouped['sRentEffective'].shift(1)
pairs['prev_lease_from'] = grouped['sLeaseFrom'].shift(1)

In [ ]:
# only keep comparable pairs and annualize the change
pairs['gap'] = (pairs['sLeaseFrom'] - pairs['prev_lease_from']).dt.days / 365.25
pairs = pairs[pairs['gap'].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive='neither')].copy()
pairs['growth_contract']  = (pairs['sRent'] / pairs['prev_rent']) ** (1 / pairs['gap']) - 1
pairs['growth_effective'] = (pairs['sRentEffective'] / pairs['prev_rent_eff']) ** (1 / pairs['gap']) - 1

In [ ]:
# naive: year-over-year change of the median contract rent of all leases (moves with the building mix)
naive_pct = leases.groupby('year')['sRent'].median().pct_change() * 100

same_unit = (pairs[pairs['year'].isin(HISTORY_YEARS)].groupby('year')
             .agg(pairs=('growth_effective', 'size'), effective_mean_pct=('growth_effective', 'mean')))
same_unit['effective_mean_pct'] *= 100
same_unit['naive_median_rent_pct'] = naive_pct.reindex(same_unit.index)
same_unit.round(2)

In [ ]:
building_mix = pd.crosstab(leases['year'], leases['sBuilding'], normalize='index').loc[list(HISTORY_YEARS)] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
years = same_unit.index
ax.plot(years, same_unit['naive_median_rent_pct'], 'o--', lw=2, color='grey', label='naive: median rent of all leases')
ax.plot(years, same_unit['effective_mean_pct'], 'o-', lw=2.5, color='tab:blue', label='same unit, effective rent')
for col, color, dy in [('naive_median_rent_pct', 'grey', 7), ('effective_mean_pct', 'tab:blue', -7)]:   # label each line at its end
    ax.annotate(f"{same_unit[col].iloc[-1]:.1f}%", (years[-1], same_unit[col].iloc[-1]), textcoords='offset points',
                xytext=(8, dy), va='center', fontsize=9, color=color, fontweight='bold')
ax.annotate(f"{same_unit.loc[2023, 'naive_median_rent_pct']:.1f}%: Le Carlyle and\nThe Met enter the mix",
            xy=(2023, same_unit.loc[2023, 'naive_median_rent_pct']), xytext=(12, 0), textcoords='offset points',
            va='center', fontsize=9, color='dimgrey')
ax.set_xticks(years)
ax.set_xlim(years[0] - 0.3, years[-1] + 0.7)
ax.set_ylim(0, same_unit['naive_median_rent_pct'].max() + 2)
ax.set_ylabel('% per year')
ax.set_title('The annual median measures the mix')
ax.legend(loc='upper left', fontsize=9, frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

building_mix.plot(kind='bar', stacked=True, ax=axes[1], width=0.8, colormap='tab10')
axes[1].set_title('Share of leases by building')
axes[1].set_xlabel('')
axes[1].set_ylabel('% of leases')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=8, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left')
axes[1].spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In 2023 the median rent of all leases jumps 10.9%, but the same units rose only 2.2%: the jump comes from new high-end buildings
entering the mix, not from tenants paying more.

## 4. Concessions
Contract rent (`sRent`) vs effective rent (`sRentEffective`, after concessions).

In [ ]:
concessions = (pairs[pairs['year'].isin(HISTORY_YEARS)].groupby('year')
               .agg(contract_pct=('growth_contract', 'mean'),
                    effective_pct=('growth_effective', 'mean'),
                    leases_with_concession_pct=('sConcession', 'mean')) * 100)
concessions['tal_pct'] = [QUEBEC_TAL_PCT[y] for y in concessions.index]

years = concessions.index
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.fill_between(years, concessions['effective_pct'], concessions['contract_pct'],
                where=concessions['contract_pct'] >= concessions['effective_pct'], interpolate=True, color='tab:orange', alpha=0.15,
                label='given back by concessions')
ax.plot(years, concessions['contract_pct'], 'o-', lw=2.5, color='tab:orange', label='contract rent')
ax.plot(years, concessions['effective_pct'], 'o-', lw=2.5, color='tab:blue', label='effective rent (after concessions)')
ax.plot(years, concessions['tal_pct'], '--', lw=1.2, color='grey', label='TAL rate', zorder=1)
last = years[-1]
for col, color in [('contract_pct', 'tab:orange'), ('effective_pct', 'tab:blue'), ('tal_pct', 'grey')]:   # label each line at its end
    ax.annotate(f"{concessions.loc[last, col]:.1f}%", (last, concessions.loc[last, col]), textcoords='offset points',
                xytext=(8, 0), va='center', fontsize=9, color=color, fontweight='bold')
ax.set_xticks(years)
ax.set_xlim(years[0] - 0.3, last + 0.6)
ax.set_ylim(0, concessions['contract_pct'].max() + 1)
ax.set_ylabel('same-unit change, % per year')
ax.set_title('Contract vs effective rent growth')
ax.legend(loc='upper left', fontsize=9, frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.show()
concessions.round(2)

## 5. Renewals vs relocations, and Ontario

In [ ]:
pairs['lease_type'] = np.where(pairs['sRenewal'] == 1, 'renewal', 'relocation')
pairs['province']   = np.where(pairs['sState'] == 'Ontario', 'Ontario', 'Quebec')

by_type = (pairs[pairs['year'].isin(HISTORY_YEARS)]
           .groupby(['year', 'lease_type'])[['growth_contract', 'growth_effective']].mean().unstack() * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, lease_type in zip(axes, ['renewal', 'relocation']):
    contract, effective = by_type[('growth_contract', lease_type)], by_type[('growth_effective', lease_type)]
    years = by_type.index
    ax.fill_between(years, effective, contract, where=contract >= effective, interpolate=True,
                    color='tab:orange', alpha=0.15, label='given back by concessions')
    ax.plot(years, contract, 'o-', lw=2.5, color='tab:orange', label='contract rent')
    ax.plot(years, effective, 'o-', lw=2.5, color='tab:blue', label='effective rent')
    for series, color in [(contract, 'tab:orange'), (effective, 'tab:blue')]:   # label each line at its end
        ax.annotate(f"{series.iloc[-1]:.1f}%", (years[-1], series.iloc[-1]), textcoords='offset points',
                    xytext=(8, 0), va='center', fontsize=9, color=color, fontweight='bold')
    ax.set_title(f"{lease_type.capitalize()}s")
    ax.set_xticks(years)
    ax.set_xlim(years[0] - 0.3, years[-1] + 0.7)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('same-unit change, % per year')
axes[0].set_ylim(0, by_type.max().max() + 1)
axes[0].legend(loc='upper left', fontsize=9, frameon=False)
fig.suptitle('Renewals vs relocations: contract vs effective rent growth')
plt.tight_layout()
plt.show()
by_type.round(2)

In [ ]:
# renewals vs the rate of their own province (never the Quebec rate for The Met)
renewals = pairs[(pairs['sRenewal'] == 1) & pairs['year'].isin(HISTORY_YEARS)].copy()
rates = {'Quebec': QUEBEC_TAL_PCT, 'Ontario': ONTARIO_GUIDELINE_PCT}
renewals['reference'] = [rates[p][y] / 100 for p, y in zip(renewals['province'], renewals['year'])]

vs_reference = (renewals.assign(above=renewals['growth_contract'] > renewals['reference'])
                .groupby(['province', 'year'])
                .agg(renewals=('above', 'size'), contract_pct=('growth_contract', 'mean'),
                     effective_pct=('growth_effective', 'mean'), reference_pct=('reference', 'mean'),
                     above_reference_pct=('above', 'mean')))
vs_reference[['contract_pct', 'effective_pct', 'reference_pct', 'above_reference_pct']] *= 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, province, rate_name in zip(axes, ['Quebec', 'Ontario'], ['TAL rate', 'Ontario guideline']):
    d = vs_reference.loc[province]
    years = d.index
    ax.fill_between(years, d['effective_pct'], d['contract_pct'], where=d['contract_pct'] >= d['effective_pct'],
                    interpolate=True, color='tab:orange', alpha=0.15, label='given back by concessions')
    ax.plot(years, d['contract_pct'], 'o-', lw=2.5, color='tab:orange', label='contract rent')
    ax.plot(years, d['effective_pct'], 'o-', lw=2.5, color='tab:blue', label='effective rent')
    ax.plot(years, d['reference_pct'], '--', lw=1.2, color='grey', label=rate_name, zorder=1)
    for col, color in [('contract_pct', 'tab:orange'), ('effective_pct', 'tab:blue'), ('reference_pct', 'grey')]:   # label each line at its end
        ax.annotate(f"{d[col].iloc[-1]:.1f}%", (years[-1], d[col].iloc[-1]), textcoords='offset points',
                    xytext=(8, 0), va='center', fontsize=9, color=color, fontweight='bold')
    ax.set_title(f"{province} renewals ({'The Met' if province == 'Ontario' else '5 buildings'})")
    ax.set_xticks(years)
    ax.set_xlim(years[0] - 0.3, years[-1] + 0.7)
    ax.legend(loc='upper left', fontsize=9, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('same-unit change, % per year')
axes[0].set_ylim(0, vs_reference[['contract_pct', 'effective_pct', 'reference_pct']].max().max() + 1)
fig.suptitle('Renewals vs the rate of their own province')
plt.tight_layout()
plt.show()
vs_reference.round(1)

# 6. External Data
We have sourced external rental survey data from CMHC ([linked here](https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/rental-market/rental-market-report-data-tables)). Now, we pull that in and integrate it with the existing data.

In [ ]:
# add rent survey data to each lease
rent_survey = pd.read_csv(bucket_file('cmhc_rental_market_survey.csv'))
rent_survey_fields = ['Vacancy', 'Average_rent', 'Estimate_rent_%change', 'Turnover']

# process dates
rent_survey['Year'] = pd.to_datetime(rent_survey['Year'], format='%Y')

# proccessing lease history
for lease in lease_history_filtered.itertuples(index=True):
  lease_date = lease.sSignDate
  lease_prior_year = lease_date - recency
  lease_bedrooms = lease.sBeds
  lease_building = lease.sBuilding
  lease_locations = building_locations[lease_building]
  for field in rent_survey_fields:
    for loc in lease_locations:
      survey_field_data = rent_survey[rent_survey['Statistic'] == field]
      survey_loc_data = survey_field_data[survey_field_data['City'] == loc]
      survey_prev_data = survey_loc_data[(survey_loc_data['Year'] <= lease_date) & (survey_loc_data['Year'] >= lease_prior_year)]
      if len(survey_prev_data) == 0:
        continue
      data = survey_prev_data.loc[survey_prev_data['Year'].idxmax(), str(lease_bedrooms)+"bed"]
      if data == '++':
        data = 0
      if pd.isna(data) or data == '**':
        continue
      lease_history_filtered.at[lease.Index, field] = data
      break

# processing active listing data
for lease in listings_filtered.itertuples(index=True):
  lease_date = lease.sAvailable
  lease_prior_year = lease_date - recency
  lease_bedrooms = lease.sBeds
  lease_building = lease.sBuilding
  lease_locations = building_locations[lease_building]
  for field in rent_survey_fields:
    for loc in lease_locations:
      survey_field_data = rent_survey[rent_survey['Statistic'] == field]
      survey_loc_data = survey_field_data[survey_field_data['City'] == loc]
      survey_prev_data = survey_loc_data[(survey_loc_data['Year'] <= lease_date) & (survey_loc_data['Year'] >= lease_prior_year)]
      if len(survey_prev_data) == 0:
        continue
      data = survey_prev_data.loc[survey_prev_data['Year'].idxmax(), str(lease_bedrooms)+"bed"]
      if data == '++':
        data = 0
      if pd.isna(data) or data == '**':
        continue
      listings_filtered.at[lease.Index, field] = data
      break

lease_history_filtered.to_csv('./codeml26/processed/processed_lease_data_external.csv', index=False)
listings_filtered.to_csv('./codeml26/processed/processed_listing_data_external.csv', index=False)

### Reconciliation with the internal trend
TAL rate, Statistics Canada CPI rent (Quebec) and asking rents (Montreal, Ottawa), CMHC same-unit rent change (Montreal),
compared with Équinoxe's same-unit growth from sections 3 and 4.

In [ ]:
import io, zipfile, urllib.request

STATCAN_ASKING_URL = 'https://www150.statcan.gc.ca/n1/tbl/csv/46100092-eng.zip'   # quarterly asking rents by CMA
STATCAN_CPI_URL    = 'https://www150.statcan.gc.ca/n1/tbl/csv/18100005-eng.zip'   # annual CPI, including rent

def statcan_table(url):
    data = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})).read()
    archive = zipfile.ZipFile(io.BytesIO(data))
    csv_name = [n for n in archive.namelist() if not n.endswith('MetaData.csv')][0]
    return pd.read_csv(archive.open(csv_name), encoding='utf-8-sig', low_memory=False)

asking_qrs = statcan_table(STATCAN_ASKING_URL)
cpi = statcan_table(STATCAN_CPI_URL)

# year-over-year change of the 2-bedroom asking rent, Q3 vs Q3
two_bed = asking_qrs[(asking_qrs['Estimates'] == 'Average asking rent') & (asking_qrs['Rental unit type'] == 'Apartment - 2 bedrooms')]
def asking_change(geo_prefix):
    q3 = two_bed[two_bed['GEO'].str.startswith(geo_prefix) & two_bed['REF_DATE'].str.endswith('-07')]
    return q3.assign(year=q3['REF_DATE'].str[:4].astype(int)).set_index('year')['VALUE'].pct_change() * 100

cpi_rent = (cpi[(cpi['Products and product groups'] == 'Rent') & (cpi['GEO'] == 'Quebec')]
            .set_index('REF_DATE')['VALUE'].pct_change() * 100)

# CMHC same-unit rent change, Montreal ("**" suppressed, "++" not significant -> missing)
survey = pd.read_csv(bucket_file('cmhc_rental_market_survey.csv'))
cmhc_change = survey[(survey['City'] == 'Montreal') & (survey['Statistic'] == 'Estimate_rent_%change')].set_index('Year')
cmhc_change = cmhc_change[['1bed', '2bed']].apply(pd.to_numeric, errors='coerce').mean(axis=1)

reconcile = pd.DataFrame(index=list(HISTORY_YEARS))
reconcile['Equinoxe_effective_pct'] = concessions['effective_pct']
reconcile['Equinoxe_contract_pct']  = concessions['contract_pct']
reconcile['TAL_pct']                = [QUEBEC_TAL_PCT[y] for y in reconcile.index]
reconcile['CPI_rent_Quebec_pct']    = cpi_rent.reindex(reconcile.index)
reconcile['CMHC_Montreal_pct']      = cmhc_change.reindex(reconcile.index)
reconcile['asking_Montreal_pct']    = asking_change('Montr').reindex(reconcile.index)
reconcile['asking_Ottawa_pct']      = asking_change('Ottawa - Gatineau (Ontario').reindex(reconcile.index)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
panels = [('Regulation and inflation', [('Equinoxe_contract_pct', 'Équinoxe contract', 'tab:orange', 'o-', 2.5),
                                        ('Equinoxe_effective_pct', 'Équinoxe effective', 'tab:blue', 'o-', 2.5),
                                        ('TAL_pct', 'TAL rate', 'grey', '--', 1.2),
                                        ('CPI_rent_Quebec_pct', 'CPI rent, Quebec', 'tab:green', ':', 1.8)]),
          ('Market', [('Equinoxe_effective_pct', 'Équinoxe effective', 'tab:blue', 'o-', 2.5),
                      ('CMHC_Montreal_pct', 'CMHC same-unit, Montreal', 'tab:brown', 's:', 1.5),
                      ('asking_Montreal_pct', 'asking rent, Montreal', 'tab:purple', '^--', 1.2),
                      ('asking_Ottawa_pct', 'asking rent, Ottawa', 'tab:pink', 'v--', 1.2)])]
for ax, (title, lines) in zip(axes, panels):
    for col, label, color, style, width in lines:
        ax.plot(reconcile.index, reconcile[col], style, color=color, lw=width, label=label)
    ax.axhline(0, color='black', lw=0.6)
    ax.set_title(title)
    ax.set_xticks(reconcile.index)
    ax.legend(fontsize=8, frameon=False, loc='upper left')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('% per year')
plt.tight_layout()
plt.show()
reconcile.round(2)

Contract rents follow the TAL rate and CPI rent; market asking rents swung from about +16% to slightly negative.
Équinoxe's effective growth stayed in a narrow 2-5% band through both: concessions absorb market swings in both directions.

## 7. Forecast and validation
We will attempt to predict each lease's annualized effective change from what is known on **January 1** of its year: the unit, its previous lease,
the last asking rent before January 1 and the year's published rate.

### 7.1 Features

In [ ]:
# read the processed lease data (section 2) and the asking history from the bucket
!gcloud storage cp gs://codeml26/processed/processed_lease_data.csv .
!gcloud storage cp gs://codeml26/jadco-data/equinoxe_asking_history.csv .

df = pd.read_csv('./processed_lease_data.csv')

In [ ]:
# convert these date columns into proper dates
for c in ['sSignDate', 'sLeaseFrom', 'sLeaseTo']:
    df[c] = pd.to_datetime(df[c], utc=True)

In [ ]:
# get lease information prior to current lease (same unit: hUnit = sPropCode + sUnitCode)
df = df.sort_values(['hUnit', 'sLeaseFrom'], kind='stable')
grouped_df = df.groupby('hUnit')

df['prev_rent_eff']   = grouped_df['sRentEffective'].shift(1)
df['prev_rent']       = grouped_df['sRent'].shift(1)
df['prev_lease_from'] = grouped_df['sLeaseFrom'].shift(1)

# only keep comparable pairs (0.5 to 2.5 years apart) and annualize the change
MIN_GAP_YEARS, MAX_GAP_YEARS = 0.5, 2.5
df['gap'] = (df['sLeaseFrom'] - df['prev_lease_from']).dt.days / 365.25
df = df[df['gap'].between(MIN_GAP_YEARS, MAX_GAP_YEARS, inclusive='neither')].copy()
df['growth_effective'] = (df['sRentEffective'] / df['prev_rent_eff']) ** (1 / df['gap']) - 1
df['growth_contract']  = (df['sRent'] / df['prev_rent']) ** (1 / df['gap']) - 1

In [ ]:
# replace asking rent at signing with the last asking rent known on january 1st of the lease year
asking = pd.read_csv('./equinoxe_asking_history.csv', usecols=['hUnit', 'sMonth', 'sAskingRent'])
asking['posted'] = pd.to_datetime(asking['sMonth'], utc=True)

jan1 = pd.to_datetime(df['sLeaseFrom'].dt.year.astype(str) + '-01-01', utc=True)
query = pd.DataFrame({'hUnit': df['hUnit'], 'jan1': jan1, 'row': df.index}).sort_values('jan1')
known = pd.merge_asof(query, asking.sort_values('posted')[['hUnit', 'posted', 'sAskingRent']],
                      left_on='jan1', right_on='posted', by='hUnit',
                      direction='backward', allow_exact_matches=False)

df['ask_jan1'] = known.set_index('row')['sAskingRent'].reindex(df.index)
df = df.drop(columns=['hAskingRent'])

In [ ]:
df['ask_over_prev_rent']     = df['ask_jan1'] / df['prev_rent']
df['ask_over_prev_rent_eff'] = df['ask_jan1'] / df['prev_rent_eff']

lease_start_year = df['sLeaseFrom'].dt.year

reference_pct = np.where(df['sState_Ontario'] == 1,
  lease_start_year.map(ONTARIO_GUIDELINE_PCT),
  lease_start_year.map(QUEBEC_TAL_PCT)
)

df['reference_rate'] = reference_pct / 100

In [ ]:
split_date      = df['sLeaseFrom'].copy()
prev_lease_from = df['prev_lease_from'].copy()
contract_growth = df['growth_contract'].copy()
prev_disc       = 1 - df['prev_rent_eff'] / df['prev_rent']
df = df.drop(columns=['sSignDate', 'sLeaseFrom', 'sLeaseTo', 'prev_lease_from', 'gap', 'growth_contract', 'hUnit'])

df = pd.get_dummies(df, dtype=int)
df.head()

### 7.2 Train and test

In [ ]:
TEST_YEAR = 2025

X = df.drop(columns=["sRentEffective", "sRent", "growth_effective"])
y = df['growth_effective']

lease_year = split_date.dt.year
known_jan1 = prev_lease_from < pd.to_datetime(lease_year.astype(str) + '-01-01', utc=True)   # previous lease known on january 1st
train_mask = lease_year < TEST_YEAR
test_mask  = (lease_year == TEST_YEAR) & known_jan1

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

In [ ]:
import os
from google.colab import userdata
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")

In [ ]:
!pip install -q tabpfn-client

from tabpfn_client import TabPFNRegressor, set_access_token
set_access_token(os.environ["TABPFN_TOKEN"])

model = TabPFNRegressor(n_estimators=8)
model.fit(X_train.astype("float32"), y_train)

In [ ]:
from sklearn.metrics import root_mean_squared_error

y_pred = model.predict(X_test.astype("float32"))
naive_pred = np.full(len(y_test), y[lease_year == TEST_YEAR - 1].mean())   # last year's mean change

print("Root mean Squared Error:", root_mean_squared_error(y_test, y_pred))
print("Naive Root mean Squared Error:", root_mean_squared_error(y_test, naive_pred))

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(
    model, X_test.astype("float32"), y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=5, random_state=42,
)

feature_importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": r.importances_mean,
}).sort_values("Importance", ascending=False)

print(feature_importance_df)

### 7.3 Backtest

In [ ]:
BACKTEST_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]   # 2023-2025 asked by the brief; earlier years widen the error sample

def backtest(year):
    train = lease_year < year
    test  = (lease_year == year) & known_jan1          # previous lease known on january 1st

    model = TabPFNRegressor(n_estimators=8)
    model.fit(X[train].astype("float32"), y[train])
    pred  = model.predict(X[test].astype("float32"))
    naive = y[lease_year == year - 1].mean()           # last year's mean change

    actual = y[test]
    return {'year': year,
            'actual_pct': 100 * actual.mean(),
            'predicted_pct': 100 * pred.mean(),
            'rmse': root_mean_squared_error(actual, pred),
            'naive_rmse': root_mean_squared_error(actual, np.full(len(actual), naive))}

backtest_results = pd.DataFrame([backtest(year) for year in BACKTEST_YEARS]).set_index('year')
backtest_results['error_pp'] = backtest_results['predicted_pct'] - backtest_results['actual_pct']

print(f"Average error on the yearly change, 2023-2025: {backtest_results.loc[2023:2025, 'error_pp'].abs().mean():.2f} pts")
print(f"Average error on the yearly change, 2020-2025: {backtest_results['error_pp'].abs().mean():.2f} pts")
backtest_results.round(4)

### 7.4 Estimate 2026

In [ ]:
N_SIMULATIONS = 10_000
RANGE_PERCENTILES = (10, 90)   # 80% range

def build_2026_rows():
    rows = pd.read_csv('./processed_lease_data.csv')
    for c in ['sSignDate', 'sLeaseFrom', 'sLeaseTo']:
        rows[c] = pd.to_datetime(rows[c], utc=True)
    rows = rows.sort_values(['hUnit', 'sLeaseFrom']).groupby('hUnit').tail(1)
    rows = rows[rows['sLeaseTo'].dt.year == FORECAST_YEAR].copy()

    rows['prev_rent_eff'] = rows['sRentEffective']
    rows['prev_rent']     = rows['sRent']
    rows['sTermSeq']     += 1

    jan1 = pd.Timestamp(f'{FORECAST_YEAR}-01-01', tz='UTC')
    rows['ask_jan1'] = rows['hUnit'].map(asking[asking['posted'] < jan1].sort_values('posted').groupby('hUnit')['sAskingRent'].last())
    rows['ask_over_prev_rent']     = rows['ask_jan1'] / rows['prev_rent']
    rows['ask_over_prev_rent_eff'] = rows['ask_jan1'] / rows['prev_rent_eff']
    rows['reference_rate'] = np.where(rows['sState_Ontario'] == 1,
                                      ONTARIO_GUIDELINE_PCT[FORECAST_YEAR], QUEBEC_TAL_PCT[FORECAST_YEAR]) / 100
    return rows

def estimate_2026():
    rows = build_2026_rows()
    model = TabPFNRegressor(n_estimators=8)
    model.fit(X.astype("float32"), y)
    rows['predicted'] = model.predict(rows.reindex(columns=X.columns).astype("float32"))
    point = 100 * rows['predicted'].mean()

    rng = np.random.default_rng(42)
    sims = point - rng.choice(backtest_results['error_pp'].to_numpy(), N_SIMULATIONS)
    low, high = np.percentile(sims, RANGE_PERCENTILES)
    return rows, point, low, high

rows_2026, point_2026, low_2026, high_2026 = estimate_2026()
print(f"{FORECAST_YEAR}: {len(rows_2026)} units | same-unit effective change {point_2026:.2f}% (80% range {low_2026:.2f}% to {high_2026:.2f}%)")

In [ ]:
rows_2026['province'] = np.where(rows_2026['sState_Ontario'] == 1, 'Ontario', 'Quebec')
print((100 * rows_2026.groupby('province')['predicted'].mean()).round(2).to_string(), '\n')
building_cols = [c for c in rows_2026.columns if c.startswith('sBuilding_')]
rows_2026['building'] = rows_2026[building_cols].idxmax(axis=1).str.replace('sBuilding_', '')
by_building = rows_2026.groupby('building')['predicted'].agg(units='size', predicted_pct='mean')
by_building['predicted_pct'] *= 100
print(by_building.round(2).to_string(), '\n')

by_apartment = rows_2026.groupby(['sBeds', 'sBaths'])['predicted'].agg(units='size', predicted_pct='mean')
by_apartment['predicted_pct'] *= 100
by_apartment.round(2)

### Assumptions
- Renewal vs relocation is not known on January 1, so 2026 is forecast for both together; sections 4-5 show how each behaved.
- The range comes from six backtest years (2020-2025) and assumes 2026 errors look like past ones.
- The model has seen one turning point (2023, overshot by about 2 points); a market turn in 2026 is a downside risk.